# Morphology-locations preview endpoint

Preview the locations a morphology-location block would generate on a single morphology, so
they can be shown in the 3D viewer before a workflow runs.

This notebook calls the declared `POST /declared/morphology-locations/preview/{entity_id}`
endpoint. The service must be running first (`make run-local`), and you need a valid bearer
token plus staging access.

The endpoint accepts an `MEModel`, a single-neuron `Circuit`, or a `CellMorphology` as
`entity_id`, and a `MorphologyLocationUnion` block as the JSON body. It returns, per generated
location, the SONATA `section_id` (0 = soma, then neurites in NEURON section order) and the
normalized `offset` along that section - the same values the block writes to
`compartment_sets.json` when the workflow runs.

In [ ]:
import requests

import obi_one as obi
from obi_auth import get_token

## Authentication

Get a staging token. The endpoint resolves the entity through entitycore, so the token and the
virtual-lab / project headers must grant access to it.

In [ ]:
token = get_token(environment="staging")
virtual_lab_id = obi.LAB_ID_STAGING_TEST
project_id = obi.PROJECT_ID_STAGING_TEST

## Build the request

`entity_id` is the morphology-carrying entity to preview against (an MEModel here). The JSON body
is one `MorphologyLocationUnion` member - a `RandomMorphologyLocations` block below, asking for 25
locations on the basal and apical dendrites (section types 3 and 4).

All parameters must be single values: parameter-sweep lists are rejected, since a preview needs
one resolved value per parameter.

In [ ]:
obi_one_api_url = "http://127.0.0.1:8100"

# An MEModel to preview against (also usable: a single-neuron circuit or a cell morphology).
entity_id = "3ba6c735-c321-4890-bb8e-60be11777567"

url = f"{obi_one_api_url}/declared/morphology-locations/preview/{entity_id}"
headers = {
    "Authorization": f"Bearer {token}",
    "Accept": "application/json",
    "Content-Type": "application/json",
}
if virtual_lab_id:
    headers["virtual-lab-id"] = virtual_lab_id
if project_id:
    headers["project-id"] = project_id

# JSON request body: a RandomMorphologyLocations block (a MorphologyLocationUnion member).
request_body = {
    "type": "RandomMorphologyLocations",
    "number_of_locations": 25,
    "section_types": [3, 4],
    "random_seed": 0,
}

## Call the endpoint and inspect the response

On success the response is `{"locations": [{"section_id": int, "offset": float}, ...]}`. We check
the count matches what was requested and that every location is well-formed: a non-negative
`section_id` and an `offset` in `[0, 1]`.

In [ ]:
response = requests.post(url, headers=headers, json=request_body)

if response.status_code == 200:
    locations = response.json()["locations"]
    print(f"Success: {len(locations)} generated locations")
    print("first location:", locations[0])

    assert len(locations) == request_body["number_of_locations"], (
        "one location per requested count"
    )
    assert all(loc["section_id"] >= 0 for loc in locations), "section ids are non-negative"
    assert all(0.0 <= loc["offset"] <= 1.0 for loc in locations), "offsets are normalized"
    # Locations were requested on dendrites only, so none should land on the soma (section 0).
    assert all(loc["section_id"] != 0 for loc in locations), "no soma locations were requested"
else:
    print(f"Error {response.status_code}: {response.text}")

## Parameter sweeps are rejected

A preview needs one resolved value per parameter, so a list-valued parameter (a sweep) returns
`422 Unprocessable Entity` rather than a preview.

In [ ]:
sweep_body = {
    "type": "RandomMorphologyLocations",
    "number_of_locations": [10, 25],
    "section_types": [3, 4],
    "random_seed": 0,
}
sweep_response = requests.post(url, headers=headers, json=sweep_body)
print(f"Status: {sweep_response.status_code}")
print(sweep_response.text)